In [4]:
from lib_linear_probing import *
from deep_models.vqvae_train import *
from deep_models.vqvae import *
import torch
from deep_models.data import S00MILDataset, make_splits, load_folds
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torch 
import pandas as pd
from sklearn.manifold import TSNE

! TO CORRECT !

In [5]:
dev_path = "/dmj/fizmed/mmarzec/licencjat_neuro/DL-NS/split_data/one_site/splits/s00_developmental.npz"
split_k = 2
num_workers = 4

In [6]:
developamental_folds = load_folds(dev_path)
_, train_idx, val_idx, test_idx = next(split for split in make_splits(developamental_folds) if split[0] == split_k)
print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

train=1078  val=129  test=146


In [7]:
trainset = S00MILDataset(train_idx, max_instances=150, train=True)
valset = S00MILDataset(val_idx, max_instances=150, train=False)
testset = S00MILDataset(test_idx, max_instances=150, train=False)


loader_kw = dict(batch_size=10, num_workers=num_workers, pin_memory=True) # type: ignore
trainloader = DataLoader(trainset, shuffle=True, **loader_kw)  # type: ignore
valloader = DataLoader(valset, shuffle=False, **loader_kw) # type: ignore
testloader = DataLoader(testset, shuffle=False, **loader_kw) # type: ignore

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 64

In [9]:
raw_tr, y_train = extract_raw_from_loader(trainloader)
raw_va, y_val   = extract_raw_from_loader(valloader)
raw_te, y_test  = extract_raw_from_loader(testloader)

raw_splits = {"train": raw_tr, "val": raw_va, "test": raw_te}

### 1. Baseline: Flattened off-diagonal raw inputs (5 * 17 * 16 = 1360 dims)


In [10]:
bags = {"raw_offdiag": {s: flatten_offdiag_bags(raw_splits[s]) for s in ("train", "val", "test")}}

### 2. Baseline: PCA on flattened off-diagonal raw inputs (64 dims, matching VAE latent_dim)

In [11]:
bags["pca"] = pca_bags(bags["raw_offdiag"], n_components=LATENT_DIM, seed=0)

### 3. Random ConvVQVAE (initialized with init_model, NO training)

In [16]:
torch.manual_seed(0)
vae_rand = ConvVQVAE(latent_channels=16,
                  code_book_size=64,
                  commitment_cost=0.25,
                  init_channels=16).to(device)
init_model(vae_rand, trainloader)   # Same Xavier/He init as before training
vae_rand.eval()
bags["vae_random_mu"] = {s: vq_encode_raw_bags(vae_rand, raw_splits[s])[0] for s in ("train", "val", "test")} #type:ignore

### 4. Trained ConvVQVAE

In [20]:
path = f"/dmj/fizmed/mmarzec/licencjat_neuro/DL-NS/deep_models/best_vqvae_fold{split_k}.pt"

vae_trained = ConvVQVAE(latent_channels=16,
                  code_book_size=64,
                  commitment_cost=0.25,
                  init_channels=16).to(device)  # match your training args
state_dict = torch.load(path, map_location=device, weights_only=True)
vae_trained.load_state_dict(state_dict)
vae_trained.eval()

bags["vae_trained_mu"] = {s: vq_encode_raw_bags(vae_trained, raw_splits[s])[0] for s in ("train", "val", "test")} #type:ignore

### 5. Trained VAE - active dimensions (threshold for mu_variance 0.1)

In [21]:
mask = active_code_mask(bags["vae_trained_mu"]["train"], thr=0.1)
print(f"Trained VAE active units: {mask.sum()} / {LATENT_DIM}")
bags["vae_trained_mu_active"] = select_dims(bags["vae_trained_mu"], mask)

Trained VAE active units: 161 / 64


In [25]:
# Continuous quantized grid (N, 400) -> drop only strictly constant columns
bags["vq_z_q"] = vq_z_q
zq_mask = active_mask(bags["vq_z_q"]["train"], thr=1e-12)
print(f"VQ active z_q dims: {zq_mask.sum()} / {len(zq_mask)}")
bags["vq_z_q_active"] = select_dims(bags["vq_z_q"], zq_mask)

VQ active z_q dims: 398 / 400


In [ ]:
representations = ["raw_offdiag", "pca", "vae_random_mu", "vae_trained_mu", "vae_trained_mu_active"]

In [ ]:
rows, artifacts = run_nn_probes(bags=bags,
                                representations=representations,
                                y_train=y_train,
                                y_val=y_val,
                                y_test=y_test,
                                return_artifacts=True,
                                abmil_variants={"ABMIL_linear": None, "ABMIL_h64": 64})
df = pd.DataFrame(rows)

In [ ]:
summary = (df.groupby(["Representation", "Method"], sort=False)[["Val AUC", "Test AUC"]].agg(["mean", "std"]).round(3))
print(summary)

## Visualization in 2-dims

In [ ]:
embeddings_2d = {rep: TSNE(n_components=2, init="pca", learning_rate="auto", random_state=0).fit_transform(artifacts[rep]["X_val"]) for rep in representations}

val_sizes = np.array([len(b) for b in bags["vae_trained_mu"]["val"]])
exam_ids = np.repeat(np.arange(len(val_sizes)), val_sizes)
window_labels = np.repeat(y_val, val_sizes)

In [ ]:
fig, axes = plot_embedding_votes(embeddings=embeddings_2d,
                                 artifacts=artifacts,
                                 exam_ids=exam_ids,
                                 window_labels=window_labels,
                                 rep_of=representations,
                                 mode="prob")
plt.show()

In [ ]:
fig, axes = plot_embedding_votes(embeddings=embeddings_2d,
                                 artifacts=artifacts,
                                 exam_ids=exam_ids,
                                 window_labels=window_labels,
                                 rep_of=representations,
                                 mode="vote")
plt.show()